# LSTM para puntuación y capitalización (TP II)

Modelo **(b)** del enunciado: una LSTM unidireccional que lee una instancia (un fragmento de texto) palabra por palabra y, **en cada palabra**, predice tres etiquetas:

| Cabeza | Clases |
|---|---|
| `punt_inicial` | 0 = nada, 1 = `¿` |
| `punt_final` | 0 = nada, 1 = `,`, 2 = `.`, 3 = `?` |
| `capitalizacion` | 0 = minúscula, 1 = Inicial, 2 = mixta, 3 = TODO MAYÚSCULA |

Es un **many-to-many sincronizado**, como el notebook de Shakespeare, pero con tres cabezas y con embeddings BERT ya calculados como entrada (no hay `nn.Embedding`).

### Notación

| Símbolo | Significado | Valor |
|---|---|---|
| `B` | instancias por batch | 32 |
| `T` | palabras de la instancia más larga del batch (el resto se rellena) | cambia en cada batch |
| `D` | features por palabra (embedding BERT) | 768 |
| `H` | tamaño del estado oculto | 128 |
| `L` | capas de LSTM apiladas | 2 |
| `K` | clases de cada cabeza | 2 / 4 / 4 |

**Archivos que necesita en la misma carpeta:** `embeddings_palabras.npy` y `etiquetas_palabras.csv` (los genera `embeddings_palabras.py`).

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

---
# Paso 1: datos a tensores

* `keep_default_na=False`: en el CSV, "sin puntuación" es un campo vacío. Sin esto pandas lo lee como `NaN` y se pierde la clase "nada".
* Las etiquetas se pasan a enteros, porque `CrossEntropyLoss` pide índices de clase.
* El split es **por instancia** (`GroupShuffleSplit`, `random_state=42`), el mismo que usa el Random Forest, así los modelos son comparables.

In [ ]:
from pathlib import Path

# La raíz del repo es la carpeta que tiene requirements.txt (sirve corriendo desde la raíz o desde notebooks/)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
SALIDAS = RAIZ / "salidas"

X = np.load(SALIDAS / "embeddings_palabras.npy")                              # (n_palabras, 768)
et = pd.read_csv(SALIDAS / "etiquetas_palabras.csv", keep_default_na=False)   # una fila por palabra, misma fila que X
print("X:", X.shape, "| etiquetas:", et.shape)

MAPA = {
    "punt_inicial":   {"": 0, "¿": 1},
    "punt_final":     {"": 0, ",": 1, ".": 2, "?": 3},
    "capitalizacion": {0: 0, 1: 1, 2: 2, 3: 3},
}
for col, mapa in MAPA.items():
    et[col + "_id"] = et[col].map(mapa)

COLS_ID = ["punt_inicial_id", "punt_final_id", "capitalizacion_id"]
print("NaN en las columnas _id:", int(et[COLS_ID].isna().sum().sum()), "(tiene que ser 0)")

# Mismo split que dividir_por_instancia() de embeddings_palabras.py
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(splitter.split(et, groups=et["instancia_id"]))
print("palabras train / test:", len(idx_train), len(idx_test))

### Validación

Se separa una parte del train (15 %, por `instancia_id`) como **validación**, para elegir hiperparámetros y la mejor época. El **test no cambia** (mismo split de antes) y no se usa hasta el final. De acá en adelante `idx_train` es el train *sin* la validación, así que los pesos de clase también se calculan sin ella.

In [ ]:
idx_train_total = idx_train                       # train completo (80 % de las instancias)
sub_tr = et.iloc[idx_train_total]
i_tr, i_val = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=0).split(sub_tr, groups=sub_tr["instancia_id"]))
idx_train, idx_val = idx_train_total[i_tr], idx_train_total[i_val]

# Ninguna instancia puede estar en dos conjuntos
ids = lambda idx: set(et.iloc[idx]["instancia_id"])
assert not (ids(idx_train) & ids(idx_val)) and not (ids(idx_train) & ids(idx_test)) and not (ids(idx_val) & ids(idx_test))
print("palabras train / val / test:", len(idx_train), len(idx_val), len(idx_test))
print("instancias train / val / test:", len(ids(idx_train)), len(ids(idx_val)), len(ids(idx_test)))

---
# Paso 2: secuencias con padding y máscara

Las instancias tienen entre 3 y 284 palabras, pero un tensor `(B, T, 768)` tiene que ser rectangular. Entonces:

* **`x`** se rellena con ceros hasta el largo de la instancia más larga del batch.
* **`y`** se rellena con `-100`, que es el `ignore_index` por defecto de `CrossEntropyLoss`: la pérdida ignora el relleno sin que hagamos nada más.
* **`mb`** (*mask batch*) es un tensor `(B, T)` de booleanos: `True` donde hay una palabra real. Lo usamos para calcular métricas.

In [ ]:
def armar_secuencias(idx_filas):
    """Una tupla (x, y) por instancia: x (T,768) float, y (T,3) long."""
    sub = et.iloc[idx_filas].sort_values(["instancia_id", "posicion"])   # palabras en orden
    secuencias = []
    for _, g in sub.groupby("instancia_id", sort=False):
        x = torch.from_numpy(X[g.index.to_numpy()])      # el índice de `et` es la fila de X
        y = torch.tensor(g[COLS_ID].values, dtype=torch.long)
        secuencias.append((x, y))
    return secuencias


def collate(batch):
    """Junta varias instancias en un batch rectangular."""
    xs, ys = zip(*batch)
    x = pad_sequence(xs, batch_first=True)                       # (B, T, 768), relleno = 0.0
    y = pad_sequence(ys, batch_first=True, padding_value=-100)   # (B, T, 3),   relleno = -100
    mask = y[:, :, 0] != -100                                    # (B, T), True = palabra real
    return x, y, mask

In [ ]:
seq_train = armar_secuencias(idx_train)
seq_val = armar_secuencias(idx_val)
seq_test = armar_secuencias(idx_test)
print("instancias train / val / test:", len(seq_train), len(seq_val), len(seq_test))


class SamplerPorLargo:
    """Arma batches de instancias de largo parecido (menos relleno = épocas más rápidas).
    mezclar=True (train): se mezclan todas las instancias, se cortan en bloques de batch_size*bloque,
      se ordenan por largo dentro de cada bloque, se forman los batches y se desordena el orden de los batches.
      Así un batch junta largos parecidos pero cada época es distinta.
    mezclar=False (val/test): orden fijo por largo (el F1 no depende del orden)."""
    def __init__(self, largos, batch_size=32, bloque=50, mezclar=True, seed=0):
        self.largos = np.asarray(largos)
        self.batch_size, self.bloque, self.mezclar = batch_size, bloque, mezclar
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return math.ceil(len(self.largos) / self.batch_size)

    def __iter__(self):
        n, bs = len(self.largos), self.batch_size
        if not self.mezclar:
            orden = np.argsort(self.largos, kind="stable")
            for j in range(0, n, bs):
                yield orden[j:j + bs].tolist()
            return
        orden = self.rng.permutation(n)
        batches = []
        for i in range(0, n, bs * self.bloque):
            bl = orden[i:i + bs * self.bloque]
            bl = bl[np.argsort(self.largos[bl], kind="stable")]
            batches += [bl[j:j + bs] for j in range(0, len(bl), bs)]
        for k in self.rng.permutation(len(batches)):
            yield batches[k].tolist()


def hacer_dl(seqs, mezclar):
    sampler = SamplerPorLargo([len(x) for x, _ in seqs], mezclar=mezclar)
    return DataLoader(seqs, batch_sampler=sampler, collate_fn=collate)


def fraccion_relleno(dl):
    """Fracción de posiciones del batch que son relleno, en una pasada completa."""
    reales = total = 0
    for idxs in dl.batch_sampler:
        largos = [len(dl.dataset[i][0]) for i in idxs]
        reales += sum(largos)
        total += max(largos) * len(largos)
    return 1 - reales / total


dl_train = hacer_dl(seq_train, mezclar=True)
dl_val = hacer_dl(seq_val, mezclar=False)
dl_test = hacer_dl(seq_test, mezclar=False)
print(f"relleno en train: {fraccion_relleno(dl_train):.1%} (antes ~46%)")

torch.manual_seed(0)
xb, yb, mb = next(iter(dl_train))
print("xb", tuple(xb.shape), "| yb", tuple(yb.shape), "| mb", tuple(mb.shape))
print("palabras reales:", mb.sum().item(), "de", mb.numel(), "posiciones")


---
# Paso 3: el modelo

### Arquitectura completa (2 capas, desplegada en el tiempo)

Entre una celda y la siguiente viajan **dos** cosas: el estado oculto `h` y el cell state `c`. Cada capa tiene su propio par `(h, c)`, que arranca en cero. La capa 2 recibe como entrada el `h_t` de la capa 1 (no su `c_t`).

```
                palabra 1        palabra 2        palabra 3                 palabra T
                x1 (768)         x2 (768)         x3 (768)                  xT (768)
                   |                |                |                         |
                   v                v                v                         v
              +---------+      +---------+      +---------+               +---------+
 h0=0 ------->| LSTM    |-h1-->| LSTM    |-h2-->| LSTM    |-h3--> ... -hT>| LSTM    |--> h_n (capa 1)
 c0=0 ------->| capa 1  |-c1-->| capa 1  |-c2-->| capa 1  |-c3--> ... -cT>| capa 1  |--> c_n (capa 1)
              | 768->128|      | 768->128|      | 768->128|               | 768->128|    (no se usan)
              +---------+      +---------+      +---------+               +---------+
                   | h1 (128)       | h2 (128)       | h3 (128)                | hT (128)
                   v                v                v                         v
               dropout 0.2 (solo entre capas y solo en train)
                   |                |                |                         |
                   v                v                v                         v
              +---------+      +---------+      +---------+               +---------+
 h0=0 ------->| LSTM    |-h1-->| LSTM    |-h2-->| LSTM    |-h3--> ... -hT>| LSTM    |--> h_n (capa 2)
 c0=0 ------->| capa 2  |-c1-->| capa 2  |-c2-->| capa 2  |-c3--> ... -cT>| capa 2  |--> c_n (capa 2)
              | 128->128|      | 128->128|      | 128->128|               | 128->128|    (no se usan)
              +---------+      +---------+      +---------+               +---------+
                   | out_1          | out_2          | out_3                   | out_T
                   |                |                |                         |      out_t = h_t de la capa 2 (128)
                   v                v                v                         v

   En CADA palabra t, el mismo trío de cabezas (mismos pesos para todos los t):

        out_t (128) --+-- Linear(128,2) --> logits punt_inicial     (2)
                      +-- Linear(128,4) --> logits punt_final       (4)
                      '-- Linear(128,4) --> logits capitalizacion   (4)      = 10 logits por palabra
```

### Qué pasa dentro de cada celda LSTM (una por capa y por palabra)

Entradas: `x_t` (o el `h_t` de la capa de abajo), `h_(t-1)` y `c_(t-1)`.

```
f_t = sigmoid(W_f . [x_t, h_(t-1)] + b_f)     compuerta de olvido
i_t = sigmoid(W_i . [x_t, h_(t-1)] + b_i)     compuerta de entrada
g_t = tanh   (W_g . [x_t, h_(t-1)] + b_g)     candidato a memoria nueva
o_t = sigmoid(W_o . [x_t, h_(t-1)] + b_o)     compuerta de salida

c_t = f_t * c_(t-1) + i_t * g_t               cell state (memoria de largo plazo)  -> a la celda siguiente
h_t = o_t * tanh(c_t)                         estado oculto  -> a la celda siguiente y a la capa de arriba / las cabezas
```

(`*` es producto elemento a elemento.) En PyTorch las 4 compuertas van apiladas: la capa 1 tiene `weight_ih` de `(512, 768)` y `weight_hh` de `(512, 128)`; la capa 2, `(512, 128)` y `(512, 128)`. Con 512 = 4 compuertas x H.

### Notas

* **El cell state `c_t` se usa en cada paso**: es la memoria que viaja de palabra en palabra y con la que se calcula `h_t`. Todo esto lo hace `nn.LSTM` internamente.
* `nn.LSTM` devuelve `out` (el `h_t` de la última capa en **todas** las palabras, `(B,T,128)`) y `(h_n, c_n)` (el `h` y el `c` solo del **último paso** de cada capa, `(L,B,128)`). Usamos `out`; `h_n` y `c_n` no los necesitamos porque predecimos una vez por palabra.
* `out` tiene un vector de 128 números **por palabra**: la palabra con el contexto que la precede.
* Los logits salen **sin softmax**: `CrossEntropyLoss` lo aplica adentro.
* `dropout` de `nn.LSTM` actúa solo **entre** capas.

In [ ]:
class PuntLSTM(nn.Module):
    def __init__(self, D=768, H=128, L=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(D, H, num_layers=L, batch_first=True, dropout=dropout)
        self.head_ini = nn.Linear(H, 2)   # nada / ¿
        self.head_fin = nn.Linear(H, 4)   # nada / , / . / ?
        self.head_cap = nn.Linear(H, 4)   # 0 / 1 / 2 / 3

    def forward(self, x, verbose=False):
        # Por dentro, la LSTM pasa (h_t, c_t) de palabra en palabra y de capa en capa.
        # Devuelve out = h_t de la última capa en TODAS las palabras, y (h_n, c_n) = h y c
        # del ÚLTIMO paso de cada capa. Solo h_n y c_n quedan sin usar.
        out, (h_n, c_n) = self.lstm(x)    # out (B,T,H) ; h_n, c_n (L,B,H)
        logits = (self.head_ini(out), self.head_fin(out), self.head_cap(out))
        if verbose:
            print(f"x          : {tuple(x.shape)}   (B, T, D)")
            print(f"out LSTM   : {tuple(out.shape)}   (B, T, H)")
            print(f"h_n, c_n   : {tuple(h_n.shape)}      (L, B, H)")
            for nombre, lg in zip(["ini", "fin", "cap"], logits):
                print(f"logits_{nombre} : {tuple(lg.shape)}   (B, T, K)")
        return logits

In [ ]:
torch.manual_seed(0)
modelo = PuntLSTM()
logits = modelo(xb, verbose=True)

print()
for nombre, p in modelo.named_parameters():
    print(f"{nombre:18s} {tuple(p.shape)}")
total_params = sum(p.numel() for p in modelo.parameters())
params_cabezas = sum(p.numel() for n, p in modelo.named_parameters() if n.startswith("head"))
print(f"\nTotal de parámetros: {total_params:,} (esperado 593,162) | cabezas: {params_cabezas:,} (esperado 1,290)")

---
# Paso 4: la pérdida

Una `CrossEntropyLoss` **por cabeza** (cada una tiene su propio vector de pesos), sumadas con peso 1:1:1.

**Pesos por clase.** Las clases raras (`¿`, `?`, mayúscula mixta) son menos del 1% de las palabras. Para que equivocarse en ellas cueste más:

```
w_c = ( N / (K * n_c) ) ^ potencia
```

* `n_c` = palabras de train de la clase `c`, `N` = total de palabras de train, `K` = cantidad de clases.
* Con `potencia = 1` es la heurística *balanced* de scikit-learn (cada clase aporta lo mismo a la pérdida). Con `potencia = 0` no hay pesos. Con `0.5` queda un punto medio.
* Se calculan **solo con train**, una vez, antes de entrenar.

**Pérdida ponderada** (la que calcula PyTorch con `reduction="mean"`), sobre las posiciones reales `S`:

```
L = sum_{i en S} w[y_i] * (-log p_i[y_i])  /  sum_{i en S} w[y_i]
```

In [ ]:
K_CLASES = {"punt_inicial_id": 2, "punt_final_id": 4, "capitalizacion_id": 4}


def pesos_clase(col_id, K, potencia=0.5):
    """Un peso por clase, calculado SOLO con las palabras de train."""
    clases_train = et.iloc[idx_train][col_id]              # clase (0..K-1) de cada palabra de train
    cuentas = np.bincount(clases_train, minlength=K)       # cuántas palabras hay de cada clase
    total = cuentas.sum()
    pesos_balanced = total / (K * cuentas)                 # cada clase aportaría lo mismo
    pesos = pesos_balanced ** potencia                     # 1.0 = balanced, 0.5 = suave, 0.0 = sin pesos
    return torch.tensor(pesos, dtype=torch.float32)


POTENCIA = 0.5
ALPHAS = (1.0, 1.0, 1.0)                                   # peso de la pérdida de cada cabeza (ini, fin, cap)
ces = []                                                   # una pérdida por cabeza, en el orden ini, fin, cap
for col_id, K in K_CLASES.items():
    pesos = pesos_clase(col_id, K, potencia=POTENCIA)
    print(f"{col_id:18s} pesos: {[round(p, 3) for p in pesos.tolist()]}")
    ces.append(nn.CrossEntropyLoss(weight=pesos))          # ignore_index=-100 ya viene por defecto

In [ ]:
def perdida(logits, y):
    """logits: tupla de 3 tensores (B,T,K). y: (B,T,3).
    Devuelve (pérdida total, lista con la pérdida de cada cabeza)."""
    perdidas_por_cabeza = []
    for i in range(3):
        lg = logits[i]                        # (B, T, K): logits de la cabeza i
        B, T, K = lg.shape
        lg_plano = lg.reshape(B * T, K)       # (B*T, K): una fila por palabra
        y_plano = y[:, :, i].reshape(B * T)   # (B*T,):   la clase real de cada palabra
        perdidas_por_cabeza.append(ces[i](lg_plano, y_plano))
    total = sum(a * l for a, l in zip(ALPHAS, perdidas_por_cabeza))   # con ALPHAS = 1,1,1 es la suma simple
    return total, perdidas_por_cabeza


logits = modelo(xb)
total, ls = perdida(logits, yb)
print("pérdidas por cabeza:", [round(l.item(), 3) for l in ls], "| total:", round(total.item(), 3))
print("ln(2), ln(4), ln(4):", [round(math.log(k), 3) for k in (2, 4, 4)], "<- valores esperados al inicio")

**Verificación:** la pérdida calculada con los `-100` en el medio tiene que coincidir con la calculada a mano solo sobre las palabras reales (`mb`).

In [ ]:
for i, nombre in enumerate(["ini", "fin", "cap"]):
    lg = logits[i]
    K = lg.shape[-1]
    con_ignore = ces[i](lg.reshape(-1, K), yb[:, :, i].reshape(-1))   # como en perdida(): con los -100
    solo_reales = ces[i](lg[mb], yb[:, :, i][mb])                     # a mano: solo donde mb es True
    print(f"{nombre}: con ignore = {con_ignore.item():.5f} | solo reales = {solo_reales.item():.5f}")

---
# Paso 5: entrenamiento y evaluación

* **Sin conjunto de validación** (decisión del equipo: hay pocos datos). El test se usa solo para *mirar* cómo va el entrenamiento. Para no ajustar nada contra el test, la cantidad de épocas se fija de antemano y se reporta la **última** época, no la mejor.
* `modelo.train()` / `modelo.eval()` activan / apagan el dropout.
* Recorte de gradiente (`clip_grad_norm_`, 1.0) y Adam, como en el notebook de clase. `lr = 1e-3` viene de los notebooks de LSTM.
* **F1 macro** por tarea, a nivel palabra, calculado solo sobre las palabras reales (`mb`).
* Las predicciones son el `argmax` de los logits (da lo mismo que el `argmax` de las probabilidades).

In [ ]:
torch.manual_seed(0)
modelo = PuntLSTM().to(device)

# Los pesos de cada pérdida tienen que estar en el mismo device que los logits
for ce in ces:
    ce.to(device)

optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

NOMBRES = ["punt_inicial", "punt_final", "capitalizacion"]


def entrenar_una_epoca(modelo, dl, optimizador):
    modelo.train()                           # activa el dropout
    suma_perdida = 0.0
    n_batches = 0
    for xb, yb, mb in dl:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = modelo(xb)                  # tupla de 3: (B,T,2), (B,T,4), (B,T,4)
        total, _ = perdida(logits, yb)       # suma de las 3 pérdidas

        optimizador.zero_grad()
        total.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        optimizador.step()

        suma_perdida += total.item()
        n_batches += 1
    return suma_perdida / n_batches          # pérdida promedio de la época


@torch.no_grad()
def evaluar(modelo, dl):
    modelo.eval()                            # apaga el dropout
    reales = [[], [], []]
    predichos = [[], [], []]
    suma_perdida = 0.0
    n_batches = 0

    for xb, yb, mb in dl:
        xb = xb.to(device)
        yb = yb.to(device)
        mb = mb.to(device)

        logits = modelo(xb)
        total, _ = perdida(logits, yb)
        suma_perdida += total.item()
        n_batches += 1

        for i in range(3):
            pred = logits[i].argmax(dim=-1)                   # (B,T): clase de mayor logit
            reales[i].append(yb[:, :, i][mb].cpu().numpy())   # solo palabras reales -> (N,)
            predichos[i].append(pred[mb].cpu().numpy())

    resultados = {"perdida": suma_perdida / n_batches}
    for i, nombre in enumerate(NOMBRES):
        y_real = np.concatenate(reales[i])
        y_pred = np.concatenate(predichos[i])
        clases = list(range(K_CLASES[nombre + "_id"]))
        resultados[nombre] = f1_score(y_real, y_pred, labels=clases, average="macro", zero_division=0)
    return resultados

### 5c. Una época de prueba

Antes de entrenar muchas épocas, verificamos que todo funciona con una sola.
**Esperado:** la pérdida de train baja de ~3.5 (la inicial). El F1 macro puede ser bajo: con una época las clases raras casi no se aprenden. Referencia del Random Forest base (aprox., promediando los F1 por clase de RESUMEN.md): 0.52 / 0.34 / 0.30.

In [ ]:
t0 = time.time()
perdida_train = entrenar_una_epoca(modelo, dl_train, optimizador)
t_entrenar = time.time() - t0

t0 = time.time()
res = evaluar(modelo, dl_val)
t_evaluar = time.time() - t0

print(f"pérdida train {perdida_train:.3f} | pérdida val {res['perdida']:.3f}")
for nombre in NOMBRES:
    print(f"  F1 macro {nombre:15s}: {res[nombre]:.3f}")
print(f"\ntiempo: entrenar {t_entrenar:.1f} s | evaluar {t_evaluar:.1f} s")

### 5d. Bucle de épocas con validación y checkpoint

Se entrena un modelo nuevo durante `N_EPOCAS` épocas (12). En cada época se evalúa en **validación** (una parte del train separada por instancia; **test no se toca**) y se calcula el score = promedio de los 3 F1 macro. Se guarda una copia de los pesos de la mejor época y al final se restauran (**checkpoint**). Cada corrida agrega una fila a `salidas/busqueda_lstm.csv`.

`correr_experimento` acepta `H, L, dropout, lr, weight_decay, potencia, alphas, seed`, así que sirve para el barrido de hiperparámetros.

In [ ]:
import copy
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

N_EPOCAS = 12
CSV_BUSQUEDA = SALIDAS / "busqueda_lstm.csv"


def correr_experimento(H=128, L=2, dropout=0.2, lr=1e-3, weight_decay=0.0, potencia=0.5,
                       alphas=(1.0, 1.0, 1.0), n_epocas=N_EPOCAS, seed=0, guardar=True, verbose=True):
    """Entrena un modelo nuevo, evalúa en val cada época y devuelve (modelo con los pesos de la mejor época, historial).
    Usa dl_train / dl_val del entorno. Cambia las globales `ces` (pesos de clase) y `ALPHAS`."""
    global ces, ALPHAS
    ces = [nn.CrossEntropyLoss(weight=pesos_clase(c, K, potencia=potencia).to(device)) for c, K in K_CLASES.items()]
    ALPHAS = tuple(alphas)
    torch.manual_seed(seed)
    dl_train.batch_sampler.rng = np.random.default_rng(seed)   # mismo orden de batches para la misma semilla

    modelo = PuntLSTM(H=H, L=L, dropout=dropout).to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=weight_decay)
    hist = {"train": [], "val": [], "score": [], **{n: [] for n in NOMBRES}}
    mejor = {"score": -1.0, "epoca": 0, "estado": None}
    t_ini = time.time()

    for ep in range(1, n_epocas + 1):
        t0 = time.time()
        hist["train"].append(entrenar_una_epoca(modelo, dl_train, opt))
        res = evaluar(modelo, dl_val)
        hist["val"].append(res["perdida"])
        for n in NOMBRES:
            hist[n].append(res[n])
        score = float(np.mean([res[n] for n in NOMBRES]))
        hist["score"].append(score)
        if score > mejor["score"]:   # checkpoint: copia de los pesos de la mejor época
            mejor.update(score=score, epoca=ep, estado={k: v.detach().clone() for k, v in modelo.state_dict().items()})
        if verbose:
            print(f"ep {ep:2d} | train {hist['train'][-1]:.3f} | val {res['perdida']:.3f} | "
                  f"F1 " + " / ".join(f"{res[n]:.3f}" for n in NOMBRES) + f" | score {score:.3f}"
                  + (" *" if mejor["epoca"] == ep else "") + f" | {time.time() - t0:.0f} s")

    modelo.load_state_dict(mejor["estado"])      # restaura la mejor época
    e = mejor["epoca"] - 1
    hist["mejor_epoca"], hist["mejor_score"] = mejor["epoca"], mejor["score"]
    if guardar:
        fila = {"H": H, "L": L, "dropout": dropout, "lr": lr, "weight_decay": weight_decay, "potencia": potencia,
                "alpha_ini": alphas[0], "alpha_fin": alphas[1], "alpha_cap": alphas[2], "seed": seed,
                "n_epocas": n_epocas, "mejor_epoca": mejor["epoca"], "score_val": round(mejor["score"], 4),
                **{"f1_" + n: round(hist[n][e], 4) for n in NOMBRES},
                "perdida_val": round(hist["val"][e], 4), "segundos": round(time.time() - t_ini)}
        pd.DataFrame([fila]).to_csv(CSV_BUSQUEDA, mode="a", header=not CSV_BUSQUEDA.exists(), index=False)
    if verbose:
        print(f"mejor época: {mejor['epoca']} | score val {mejor['score']:.3f}")
    return modelo, hist


modelo, hist = correr_experimento()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ep = range(1, len(hist["train"]) + 1)
ejes[0].plot(ep, hist["train"], label="train"); ejes[0].plot(ep, hist["val"], label="val")
ejes[0].set_title("Pérdida"); ejes[0].set_xlabel("época"); ejes[0].legend()
for n in NOMBRES:
    ejes[1].plot(ep, hist[n], label=n)
ejes[1].plot(ep, hist["score"], "k--", label="score (promedio)")
for a in ejes:
    a.axvline(hist["mejor_epoca"], color="gray", ls=":")   # época elegida por el checkpoint
ejes[1].set_title("F1 macro (val)"); ejes[1].set_xlabel("época"); ejes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def predecir(modelo, dl):
    """Devuelve (reales, predichos): listas de 3 arrays con solo las palabras reales."""
    modelo.eval()
    reales, predichos = [[], [], []], [[], [], []]
    for xb, yb, mb in dl:
        logits = modelo(xb.to(device))
        yb, mb = yb.to(device), mb.to(device)
        for i in range(3):
            reales[i].append(yb[:, :, i][mb].cpu().numpy())
            predichos[i].append(logits[i].argmax(-1)[mb].cpu().numpy())
    return [np.concatenate(r) for r in reales], [np.concatenate(p) for p in predichos]


NOMBRES_CLASES = {"punt_inicial": ["nada", "¿"], "punt_final": ["nada", ",", ".", "?"], "capitalizacion": ["0", "1", "2", "3"]}
reales, predichos = predecir(modelo, dl_val)
for i, n in enumerate(NOMBRES):
    k = K_CLASES[n + "_id"]
    print(f"=== {n} ===")
    print(classification_report(reales[i], predichos[i], labels=list(range(k)),
                                target_names=NOMBRES_CLASES[n], digits=3, zero_division=0))
    print("predicciones por clase :", np.bincount(predichos[i], minlength=k).tolist())
    print("reales por clase       :", np.bincount(reales[i], minlength=k).tolist())
    print()

### Etapa A: capas (`L`) y tamaño oculto (`H`)

Resto de hiperparámetros fijos (dropout 0,2, lr 1e-3, potencia 0,5, alphas 1,1,1, semilla 0). Se saltean las combinaciones que ya están en `busqueda_lstm.csv`, así que se puede cortar y retomar. La selección es por score en **val**.

In [ ]:
etapa = "A"
combinaciones = [(L, H) for L in (1, 2, 3) for H in (128, 256)]
N_EPOCAS = 12
CSV_BUSQUEDA = SALIDAS / "busqueda_lstm.csv"


hechas = set()
if CSV_BUSQUEDA.exists():
    prev = pd.read_csv(CSV_BUSQUEDA)
    prev = prev[(prev.dropout == 0.2) & (prev.lr == 0.001) & (prev.weight_decay == 0) & (prev.potencia == 0.5)
                & (prev.alpha_ini == 1) & (prev.alpha_fin == 1) & (prev.alpha_cap == 1) & (prev.seed == 0) & (prev.n_epocas == N_EPOCAS)]
    hechas = set(zip(prev.L, prev.H))

for L, H in combinaciones:
    if (L, H) in hechas:
        print(f"L={L} H={H}: ya está en el CSV, se saltea")
        continue
    t0 = time.time()
    _, h = correr_experimento(L=L, H=H, verbose=False)
    print(f"L={L} H={H}: mejor época {h['mejor_epoca']:2d} | score val {h['mejor_score']:.3f} | "
          f"F1 {h['punt_inicial'][h['mejor_epoca']-1]:.3f} / {h['punt_final'][h['mejor_epoca']-1]:.3f} / "
          f"{h['capitalizacion'][h['mejor_epoca']-1]:.3f} | {(time.time()-t0)/60:.1f} min")

res_A = pd.read_csv(CSV_BUSQUEDA)
res_A = res_A[(res_A.dropout == 0.2) & (res_A.lr == 0.001) & (res_A.potencia == 0.5) & (res_A.seed == 0)]
print()
print(res_A.sort_values("score_val", ascending=False)[["L", "H", "mejor_epoca", "score_val", "f1_punt_inicial", "f1_punt_final", "f1_capitalizacion", "perdida_val", "segundos"]].to_string(index=False))

### Etapa C: `potencia` y alphas

Base fija: L=2, H=128, dropout 0,2, lr 1e-3, semilla 0 (la etapa A mostró que L y H casi no importan). Primero se compara `potencia` ∈ {0, 0,5, 1} con alphas 1,1,1; después, con la mejor `potencia`, se prueban alphas que dan más peso a `punt_final` y `capitalizacion`. Se retoma desde el CSV si se corta.

In [ ]:
BASE = dict(H=128, L=2, dropout=0.2, lr=1e-3, weight_decay=0.0, seed=0)


def correr_si_falta(potencia, alphas):
    """Corre la config (BASE + potencia + alphas) solo si no está ya en el CSV; devuelve su fila de resultados."""
    def buscar():
        if not CSV_BUSQUEDA.exists():
            return None
        d = pd.read_csv(CSV_BUSQUEDA)
        m = ((d.H == BASE["H"]) & (d.L == BASE["L"]) & (d.dropout == BASE["dropout"]) & (d.lr == BASE["lr"])
             & (d.weight_decay == BASE["weight_decay"]) & (d.seed == BASE["seed"]) & (d.n_epocas == N_EPOCAS)
             & (d.potencia == potencia) & (d.alpha_ini == alphas[0]) & (d.alpha_fin == alphas[1]) & (d.alpha_cap == alphas[2]))
        return d[m].iloc[-1] if m.any() else None
    fila = buscar()
    if fila is None:
        correr_experimento(potencia=potencia, alphas=alphas, verbose=False, **BASE)
        fila = buscar()
    print(f"potencia={potencia} alphas={alphas}: score val {fila.score_val:.3f} | F1 {fila.f1_punt_inicial:.3f} / "
          f"{fila.f1_punt_final:.3f} / {fila.f1_capitalizacion:.3f} | época {int(fila.mejor_epoca)}")
    return fila


# C1: potencia con alphas 1,1,1
filas = {pot: correr_si_falta(pot, (1.0, 1.0, 1.0)) for pot in (0.0, 0.5, 1.0)}
mejor_pot = max(filas, key=lambda k: filas[k].score_val)
print("-> mejor potencia:", mejor_pot)

# C2: alphas con la mejor potencia
for alphas in [(1.0, 2.0, 1.0), (1.0, 1.0, 2.0), (1.0, 2.0, 2.0)]:
    correr_si_falta(mejor_pot, alphas)

In [ ]:
res_C = pd.read_csv(CSV_BUSQUEDA)
res_C = res_C[(res_C.H == 128) & (res_C.L == 2) & (res_C.dropout == 0.2) & (res_C.lr == 0.001) & (res_C.seed == 0)]
print(res_C.sort_values("score_val", ascending=False)[["potencia", "alpha_ini", "alpha_fin", "alpha_cap", "mejor_epoca",
      "score_val", "f1_punt_inicial", "f1_punt_final", "f1_capitalizacion", "segundos"]].to_string(index=False))

### Etapa B: dropout, learning rate y weight decay

Base: L=2, H=128, `potencia` 0,5, alphas (1,1,1), semilla 0 (score val 0,591). Se varía **un eje a la vez** (dropout, lr, weight decay) y al final se corre una vez la combinación de lo mejor de cada eje. Se retoma desde el CSV si se corta.

- **dropout**: probabilidad de apagar unidades entre las dos capas de la LSTM (regulariza).
- **lr**: tamaño del paso de Adam.
- **weight decay**: penalización que empuja los pesos hacia cero (regularización L2).

In [ ]:
CFG_BASE = dict(H=128, L=2, dropout=0.2, lr=1e-3, weight_decay=0.0, potencia=0.5, alphas=(1.0, 1.0, 1.0), seed=0)


def correr_cfg(**cambios):
    """Corre CFG_BASE con los cambios indicados, salvo que ya esté en el CSV. Devuelve la fila de resultados."""
    cfg = {**CFG_BASE, **cambios}

    def buscar():
        if not CSV_BUSQUEDA.exists():
            return None
        d = pd.read_csv(CSV_BUSQUEDA)
        m = (d.n_epocas == N_EPOCAS) & (d.seed == cfg["seed"])
        for col in ("H", "L", "dropout", "lr", "weight_decay", "potencia"):
            m &= np.isclose(d[col], cfg[col])
        for col, a in zip(("alpha_ini", "alpha_fin", "alpha_cap"), cfg["alphas"]):
            m &= np.isclose(d[col], a)
        return d[m].iloc[-1] if m.any() else None

    fila = buscar()
    if fila is None:
        correr_experimento(verbose=False, **cfg)
        fila = buscar()
    desc = ", ".join(f"{k}={v}" for k, v in cambios.items()) or "base"
    print(f"{desc:28s}: score val {fila.score_val:.3f} | F1 {fila.f1_punt_inicial:.3f} / {fila.f1_punt_final:.3f} / "
          f"{fila.f1_capitalizacion:.3f} | época {int(fila.mejor_epoca)}")
    return fila


ejes = {"dropout": [0.1, 0.2, 0.3, 0.5], "lr": [3e-4, 1e-3, 3e-3], "weight_decay": [0.0, 1e-5, 1e-4]}
mejores = {}
for eje, valores in ejes.items():
    filas = {v: correr_cfg(**{eje: v}) for v in valores}
    mejores[eje] = max(filas, key=lambda v: filas[v].score_val)
    print(f"-> mejor {eje}: {mejores[eje]}")

print()
print("combinación de lo mejor de cada eje:")
fila_comb = correr_cfg(**mejores)

In [ ]:
res_B = pd.read_csv(CSV_BUSQUEDA)
res_B = res_B[(res_B.H == 128) & (res_B.L == 2) & (res_B.potencia == 0.5) & (res_B.seed == 0)
              & (res_B.alpha_ini == 1) & (res_B.alpha_fin == 1) & (res_B.alpha_cap == 1)]
print(res_B.sort_values("score_val", ascending=False)[["dropout", "lr", "weight_decay", "mejor_epoca", "score_val",
      "f1_punt_inicial", "f1_punt_final", "f1_capitalizacion", "perdida_val", "segundos"]].to_string(index=False))

### Etapa D: umbral por clase (sobre las probabilidades)

En vez de `argmax` sobre las probabilidades `p`, se hace `argmax(p * w)` con un factor `w_k` por clase (`w_0 = 1`; un factor mayor que 1 hace que la clase se prediga más). Es un ajuste **posterior**: no reentrena nada. Los factores se buscan sobre **validación** maximizando el F1 macro de cada cabeza (búsqueda por coordenadas sobre una grilla logarítmica).

**Chequeo contra el optimismo:** optimizar y medir en el mismo conjunto infla el resultado. Por eso se parte val en dos mitades (por instancia): se optimiza en una y se mide en la otra, y viceversa. La mejora que importa es la de las mitades *no vistas*. Los factores finales se ajustan con todo val. El test no se usa.

Primero se reentrena el modelo elegido en las etapas A–B–C (L=2, H=128, dropout 0,2, lr 3e-3, `potencia` 0,5, alphas 1,1,1), ~5 min.

In [ ]:
import json

CFG_D = dict(H=128, L=2, dropout=0.2, lr=3e-3, weight_decay=0.0, potencia=0.5, alphas=(1.0, 1.0, 1.0), seed=0)
modelo_D, hist_D = correr_experimento(guardar=False, **CFG_D)   # guardar=False: la fila ya está en el CSV


@torch.no_grad()
def probs_y_grupos(modelo, dl):
    """Probabilidades (softmax) por cabeza, clases reales y el índice de instancia de cada palabra (solo palabras reales)."""
    modelo.eval()
    P, Y, G = [[], [], []], [[], [], []], []
    for idxs, (xb, yb, mb) in zip(dl.batch_sampler, dl):   # dl_val no mezcla: el orden de los dos recorridos coincide
        logits = modelo(xb.to(device))
        mb_d = mb.to(device)
        for i in range(3):
            P[i].append(torch.softmax(logits[i], dim=-1)[mb_d].cpu().numpy())
            Y[i].append(yb[:, :, i][mb].numpy())
        G.append(torch.tensor(idxs).unsqueeze(1).expand_as(mb)[mb].numpy())
    return [np.concatenate(p) for p in P], [np.concatenate(y) for y in Y], np.concatenate(G)


def f1_macro(y, pred, K):
    return f1_score(y, pred, labels=list(range(K)), average="macro", zero_division=0)


def buscar_factores(P, y, K, pasadas=2, grilla=np.exp(np.linspace(-3, 3, 31))):
    """Factores w (w[0]=1) que maximizan el F1 macro de argmax(P * w), por búsqueda por coordenadas."""
    w = np.ones(K)
    mejor = f1_macro(y, (P * w).argmax(1), K)
    for _ in range(pasadas):
        for k in range(1, K):
            for f in grilla:
                w2 = w.copy()
                w2[k] = f
                s = f1_macro(y, (P * w2).argmax(1), K)
                if s > mejor + 1e-9:
                    mejor, w = s, w2
    return w, mejor


P_val, Y_val, G_val = probs_y_grupos(modelo_D, dl_val)
rng = np.random.default_rng(0)
grupos = rng.permutation(np.unique(G_val))
en_A = np.isin(G_val, grupos[: len(grupos) // 2])
mitades = [(en_A, ~en_A), (~en_A, en_A)]          # (donde se optimiza, donde se mide)

resultados_D, factores_D = {}, {}
for i, n in enumerate(NOMBRES):
    K = K_CLASES[n + "_id"]
    antes, despues = [], []
    for opt_m, med_m in mitades:
        w, _ = buscar_factores(P_val[i][opt_m], Y_val[i][opt_m], K)
        antes.append(f1_macro(Y_val[i][med_m], P_val[i][med_m].argmax(1), K))
        despues.append(f1_macro(Y_val[i][med_m], (P_val[i][med_m] * w).argmax(1), K))
    w_final, f1_opt = buscar_factores(P_val[i], Y_val[i], K)
    factores_D[n] = w_final.tolist()
    resultados_D[n] = (np.mean(antes), np.mean(despues))
    print(f"{n:15s}: F1 sin umbral {np.mean(antes):.3f} -> con umbral {np.mean(despues):.3f} (en mitades no vistas) | "
          f"factores finales {np.round(w_final, 2).tolist()}")

print()
print("score val (promedio) sin umbral:", round(np.mean([r[0] for r in resultados_D.values()]), 3),
      "| con umbral:", round(np.mean([r[1] for r in resultados_D.values()]), 3))
with open(SALIDAS / "umbrales_etapa_D.json", "w", encoding="utf8") as f:
    json.dump({"config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in CFG_D.items()},
               "factores": factores_D, "n_epocas": N_EPOCAS, "mejor_epoca": hist_D["mejor_epoca"]}, f, indent=1)
print("factores guardados en", SALIDAS / "umbrales_etapa_D.json")

---
### Lo que sigue
* ~~5d: bucle de épocas con validación y checkpoint~~ (hecho; `correr_experimento` sirve para los barridos de L, H, dropout, lr, weight decay, potencia y alphas).
* Comparar `POTENCIA` = 0 / 0.5 / 1 (y quizás `H` = 128 contra 256).
* Ajuste de umbral sobre las probabilidades, si las clases raras siguen flojas.
* LSTM **bidireccional** (necesita `pack_padded_sequence`) como modelo aparte.
* Sumar los datos de conferencias.